# Qiskit Fall Fest 2026, Challenge 2: Max-Cut with QAOA

Opens Tue, Nov 17, 2026 at 6:00 PM. Closes Sun, Nov 22, 2026 at 11:59 PM. Both times are Clemson time (US Eastern).

**The task.** Find the best cut of a 12-node graph from samples taken on real hardware.

Start from Tuesday's notebook. Its Labs 2 and 3 built a QAOA circuit for a graph, ran it, and scored the bitstrings that came back. QAOA, the quantum approximate optimization algorithm, is Tuesday's recipe for such a problem: a circuit with a few angles that makes good answers more likely than bad ones. This notebook gives you a new graph and asks for more than one lucky sample: your score comes from how good all of your samples are.

How to use this notebook:

- Run the three code cells of the Setup section first.
- Read "The task in detail". It says what counts and how a run is scored.
- Run the cells under "The tools". Their code is complete. They end with a practice run on a simulator, a program that imitates a quantum computer. It runs on the computer that runs this notebook.
- Do your own work under "Your attempt". A practice run needs no account. A run that counts needs a real quantum computer.
- Turn in the line that the notebook prints. The steps are under "Turn it in".

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## The task in detail

### The graph and its cuts

The graph has 12 nodes and 17 edges. It is a different graph from Tuesday's, so Tuesday's answer does not carry over. As on Tuesday, a cut splits the nodes into two sides, and its value is the number of edges whose two ends are on different sides. A bitstring of 12 characters describes a cut: the bit of node $i$ says which side the node is on, and Qiskit prints node 0 as the last character.

Random guessing gives a cut value of 8.5 on average, half of the 17 edges. The best cut of this graph has the value 16.

### What you run

The circuit is the QAOA circuit of Tuesday's Lab 2 for this graph: `qaoa_ansatz(cost, reps=p)` with $p$ layers, measured on all 12 qubits. You choose the number of layers, 1, 2 or 3, and the angles. One layer has two angles, $\beta$ and $\gamma$. Each further layer adds two more.

You send one job with one circuit and 4,096 shots. A job is one request to run circuits, and a shot is one run of the circuit, which returns one bitstring.

### Two numbers from your samples

- **The best sample.** The largest cut value among your 4,096 bitstrings. It has to be 16, the best cut of the graph.
- **The average.** The mean cut value over all 4,096 shots. This is the number that is scored.

Why the average? The graph has 4,096 possible bitstrings and two of them are best cuts. So 4,096 shots of pure chance contain a best cut about 86 times in 100, and the best sample alone says almost nothing about your circuit. The average does: chance gives 8.5, and a circuit that leans toward large cuts gives more.

### The rules

- A real quantum computer. The first choice is an IBM processor through your own IBM Quantum account. The second option is a device reached through qBraid. A simulator does not count.
- The circuit is `qaoa_ansatz` for this graph with 1, 2 or 3 layers. Transpiling may choose the qubits, the routing and the optimization level, and you may remove gates that do nothing, but the transpiled circuit must do exactly what the QAOA circuit does: no approximation, and no gates added.
- The options of the Sampler stay as they are. Every shot counts, and no bitstring is changed after the measurement.
- Private jobs are not accepted, because an organizer may ask you to show the job.
- The job is created after the challenge opens and before it closes.

Some words in these rules. To transpile is to rewrite a circuit in the gates of one processor and to place it on that processor's qubits. Routing is the part of that work that adds gates wherever two qubits that must interact are not neighbors on the chip. The optimization level, from 0 to 3, says how hard the transpiler works to shorten the circuit. The Sampler is the Qiskit tool that runs a circuit for a number of shots and returns the counts. A private job is a job marked so that its circuits are deleted once it has run and its results can be read only once.

### Your score

A run is valid when your samples contain a best cut, with the value 16, and their average is not above what your angles give without noise, which `exact_average` in "The tools" computes. A valid run with the average $A$ scores

$$\text{score} = \min\left(100, \; \left\lfloor 100 \times \frac{A - 8.5}{11.5 - 8.5} \right\rfloor\right)$$

and 0 if $A$ is 8.5 or less. The brackets mean "rounded down". An average of 10.0 scores 50, 11.0 scores 83, and 11.5 or more scores 100. You may turn in a line as often as you like, and your best valid line counts. Repeating the same job does not help much: the average of 4,096 shots moves by about 0.03 from one run to the next, which is one point. How the event uses the scores is explained on the event page.

### What to expect

On Nov 14, 2026, Tuesday's method, unchanged, was run twice for this graph on the IBM processor ibm_fez: one layer, Tuesday's two angles, optimization level 1. The averages were 11.22 and 11.27, which is a score of 90 to 92. The last points take a decision of your own.

The practice runs in this notebook use a simulated processor. For this circuit its averages came out close to those two real runs. A real processor changes from day to day and from qubit to qubit, though, so use the practice run to test your code and to compare your own choices, and do not read a score from it.

## The tools

The code in this part is complete. Run each cell and read what it does. You will call these functions in "Your attempt".

The first cell holds the graph and the scoring of bitstrings.

- `cut_value(bits, edges)` is Tuesday's function: the cut value of one bitstring.
- `average_cut(counts, edges)` is the answer to Tuesday's third exercise: the average cut value over all the shots in a dictionary of counts.
- `histogram(counts)` returns 17 numbers: how many shots have the cut value 0, 1, 2 and so on up to 16.
- The last lines try every one of the 4,096 possible bitstrings, as Tuesday's notebook did, and keep the best ones.

In [ ]:
import numpy as np

SHOTS = 4096            # the rules fix this number
TARGET = 11.5           # the average that earns the full score
n = 12                  # one qubit for each node

# The challenge graph: 12 nodes, 17 edges.
edges = [(0, 2), (0, 8), (0, 9), (1, 5), (1, 7), (2, 4), (2, 11), (3, 6), (3, 9),
         (3, 10), (4, 7), (4, 8), (5, 6), (5, 9), (6, 11), (7, 10), (10, 11)]


def cut_value(bits, edges):
    z = bits[::-1]    # qubit 0 is the last bit
    return sum(z[i] != z[j] for i, j in edges)


def average_cut(counts, edges):
    """Return the average cut value over all the shots in counts."""
    total_cut = sum(cut_value(bits, edges) * shots for bits, shots in counts.items())
    return total_cut / sum(counts.values())


def histogram(counts):
    """Return 17 numbers: the shots with a cut value of 0, 1, 2, ..., 16."""
    shots_at = [0] * 17
    for bits, shots in counts.items():
        shots_at[cut_value(bits, edges)] += shots
    return shots_at


# The cut value of every possible bitstring, and the best ones.
ALL_CUTS = np.array([cut_value(format(k, "012b"), edges) for k in range(2 ** n)])
BEST_VALUE = int(ALL_CUTS.max())
BEST_BITSTRINGS = [format(k, "012b") for k in range(2 ** n) if ALL_CUTS[k] == BEST_VALUE]
print("best possible cut:", BEST_VALUE, "| reached by", len(BEST_BITSTRINGS), "bitstrings")
print("average over all bitstrings (random guessing):", ALL_CUTS.mean())

The second cell builds the circuit and computes what it would give without noise.

- `build_circuit(layers)` is Tuesday's Lab 2: the cost operator of the graph, then `qaoa_ansatz` with the number of layers you ask for, then a measurement of every qubit. The angles stay open.
- The angles of a circuit are listed in a fixed order: first every $\beta$, then every $\gamma$. For one layer the list is `[beta, gamma]`, as on Tuesday. For two layers it is `[beta0, beta1, gamma0, gamma1]`.
- `exact_average(angles, layers)` computes the average cut that the circuit would give on a perfect quantum computer. It uses the exact state, so it needs no shots and no account, and it takes a fraction of a second. Use it to choose your angles before you spend time on a real processor.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp, Statevector

cost = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], 1.0) for i, j in edges],
    num_qubits=n)


def build_circuit(layers):
    """The QAOA circuit for the challenge graph with open angles, measured on every qubit."""
    qc = qaoa_ansatz(cost, reps=layers)
    qc.measure_all()
    return qc


def exact_probabilities(angles, layers):
    """The probability of every bitstring for a QAOA circuit without noise."""
    state = Statevector(qaoa_ansatz(cost, reps=layers).assign_parameters(angles))
    return state.probabilities()


def exact_average(angles, layers):
    """The average cut that the circuit would give on a perfect quantum computer."""
    return float(exact_probabilities(angles, layers) @ ALL_CUTS)


tuesday = [1.20, 0.32]                      # Tuesday's angles: [beta, gamma]
print("parameters of a one-layer circuit:", [p.name for p in build_circuit(1).parameters])
print("parameters of a two-layer circuit:", [p.name for p in build_circuit(2).parameters])
print("one layer with Tuesday's angles, without noise:", round(exact_average(tuesday, 1), 3))

Tuesday's angles work for this graph too: without noise one layer gives an average of 11.848. That is above the target of 11.5, but a real processor loses part of it to errors, as "What to expect" showed.

The third cell is the self-check, `agreement`. It works out, without noise and without a quantum computer, what a transpiled circuit does, and compares it with the QAOA circuit for your angles. The result is a number between 0 and 1, and 1 means that the two give every bitstring with the same probability. A circuit that was transpiled correctly reaches 1. The organizers use the same function when they check a submission. You do not need to follow its code to use it.

In [ ]:
MAX_REPLAY_QUBITS = 24      # the largest number of qubits the self-check simulates
NOT_ALLOWED = ("reset", "initialize", "store", "if_else", "while_loop", "for_loop", "switch_case", "box")


def result_probabilities(circuit):
    """The probability of every result of a circuit that ends in measurements, computed without noise.

    Returns None for a circuit that the challenge does not allow or that is too large to simulate.
    """
    read_by = {}                                 # result bit -> the qubit it reads
    finished = set()                             # qubits that were measured already
    gates = []
    for inst in circuit.data:
        name = inst.operation.name
        qubits = [circuit.find_bit(q).index for q in inst.qubits]
        if name in ("barrier", "delay"):
            continue
        if name in NOT_ALLOWED or finished & set(qubits):
            return None                          # no reset, no feedback, nothing after a qubit's measurement
        if name == "measure":
            finished.add(qubits[0])
            read_by[circuit.find_bit(inst.clbits[0]).index] = qubits[0]
        else:
            gates.append((inst.operation, qubits))
    if circuit.num_clbits != n or len(finished) != n or sorted(read_by) != list(range(n)):
        return None                              # exactly 12 measurements, one for each result bit
    touched = sorted(finished | {q for _, qubits in gates for q in qubits})
    if len(touched) > MAX_REPLAY_QUBITS:
        return None
    place = {q: k for k, q in enumerate(touched)}
    body = QuantumCircuit(len(touched))          # the same gates on a small register, without the measurements
    for operation, qubits in gates:
        body.append(operation, [place[q] for q in qubits])
    try:
        state = Statevector(body)
    except Exception:
        return None                              # an instruction that is not a gate
    return state.probabilities([place[read_by[k]] for k in range(n)])


def agreement(circuit, angles, layers):
    """How well a circuit agrees with the QAOA circuit for these angles: 1 means the same results, 0 means none in common."""
    got = result_probabilities(circuit)
    if got is None:
        return 0.0
    want = exact_probabilities(angles, layers)
    return float(np.sum(np.sqrt(got * want)) ** 2)

The fourth cell turns counts into the numbers of the challenge and writes the line you turn in.

- `problems(counts, angles, layers)` lists the reasons why a run is not valid. An empty list means that it is valid.
- `score(counts)` is the score formula from above, computed from whole numbers.
- `submission_line(...)` writes the line.

In [ ]:
import qiskit
import qiskit_ibm_runtime


def problems(counts, angles, layers):
    """Return the list of reasons why a run is not valid. An empty list means the run is valid."""
    found = []
    if sum(counts.values()) != SHOTS:
        found.append("the run needs exactly " + str(SHOTS) + " shots")
    if any(len(bits) != n for bits in counts):
        found.append("every result needs " + str(n) + " characters, one for each node")
        return found
    if histogram(counts)[BEST_VALUE] < 1:
        found.append("no sample is a best cut, with the value " + str(BEST_VALUE))
    if layers not in (1, 2, 3) or len(angles) != 2 * layers:
        found.append("the circuit needs 1, 2 or 3 layers and two angles for each layer")
    elif average_cut(counts, edges) > exact_average(angles, layers):
        found.append("the average is above what these angles give without noise")
    return found


def score(counts):
    """The score of a valid run, from the total cut value of all shots."""
    total_cut = sum(value * shots for value, shots in enumerate(histogram(counts)))
    gain = 2 * total_cut - 17 * SHOTS            # twice (total - 8.5 * shots), kept in whole numbers
    return max(0, min(100, (100 * gain) // (6 * SHOTS)))     # 6 * shots is twice (11.5 - 8.5) * shots


def submission_line(device, job_id, counts, angles, layers, created_utc, second_option=None):
    """The one line of text that you turn in."""
    best = max(counts, key=lambda b: cut_value(b, edges))
    on_best = [counts.get(bits, 0) for bits in BEST_BITSTRINGS]
    versions = "qiskit=" + qiskit.__version__ + " "
    versions += ("qbraid=" + second_option) if second_option else ("runtime=" + qiskit_ibm_runtime.__version__)
    return ("QFF26-C2 device=" + str(device) + " job=" + str(job_id) + " shots=" + str(SHOTS) + " layers=" + str(layers)
            + " angles=" + ",".join(format(float(a), ".4f") for a in angles) + " best=" + best
            + " opt=" + ",".join(str(c) for c in on_best) + " hist=" + ",".join(str(c) for c in histogram(counts))
            + " mean=" + format(average_cut(counts, edges), ".3f") + " utc=" + created_utc + " " + versions)

The fifth cell loads the transpiler and the Sampler, which the cells of "Your attempt" need whatever the switches say. Its `patch_target` and `CELL` matter only on the simulated processor. As in Tuesday's notebook, the transpiler is kept there on one ring of 12 qubits of the simulated chip, because a simulation slows down with every extra qubit that a circuit touches. Both are copied from Tuesday's notebook.

In [ ]:
from qiskit.transpiler import Target, generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

# One 12-qubit ring of the simulated processor, as in Tuesday's notebook.
CELL = [101, 102, 103, 104, 105, 116, 117, 121, 122, 123, 124, 125]


def patch_target(backend, qubits):
    """Return the backend's rule book with the two-qubit gate allowed only inside `qubits`."""
    keep = set(qubits)
    patch = Target(num_qubits=backend.num_qubits)
    for name in backend.target.operation_names:
        rows = {where: props for where, props in backend.target[name].items()
                if where is not None and (len(where) == 1 or set(where) <= keep)}
        if rows:
            patch.add_instruction(backend.target.operation_from_name(name), rows)
    return patch

The last cell of this part is a practice run with Tuesday's method: one layer, Tuesday's angles, optimization level 1. It always runs on the simulated processor, whatever the switches in the Setup section say, so it needs no account and sends nothing. The cell takes about ten seconds.

In [ ]:
practice_pm = generate_preset_pass_manager(optimization_level=1, target=patch_target(SIMULATED_BACKEND, CELL),
                                           seed_transpiler=7)
practice_isa = practice_pm.run(build_circuit(1)).assign_parameters(tuesday)
print("two-qubit gates after transpiling:", practice_isa.count_ops().get("cz"))
print("self-check, agreement with the QAOA circuit:", round(agreement(practice_isa, tuesday, 1), 6))

practice_counts = SamplerV2(mode=SIMULATED_BACKEND).run([practice_isa], shots=SHOTS).result()[0].data.meas.get_counts()
print("best sample:", max(cut_value(bits, edges) for bits in practice_counts), "| average cut:", round(average_cut(practice_counts, edges), 3))
print("not valid because:" if problems(practice_counts, tuesday, 1) else "these samples would be valid",
      *problems(practice_counts, tuesday, 1))
print("This was a practice run on the simulated processor. It is not a submission.")

On the simulated processor the average comes out near 11.3, and the best sample is 16. The self-check prints 1.0: the transpiled circuit does what the QAOA circuit does.

## Your attempt

From here on the work is yours. The next cell holds your choices. Change them, then run the cells of this part in order.

- **How many layers.** `layers`, 1, 2 or 3. More layers can give a higher average without noise, and they make the circuit longer, which costs on a real processor.
- **The angles.** `my_angles`, in the order every $\beta$ first, then every $\gamma$. For one layer, Tuesday's pair is close to the best. A second layer needs four new angles. Tuesday's pair used twice is where a search can start, not an answer: it gives less than one layer does. `exact_average` tells you in a fraction of a second what a choice of angles is worth without noise, so you can search with it. Tuesday's "Try it" cell showed how an optimizer searches for the lowest value of a function. An optimizer looks for the lowest value and you want the highest average, so give it the negative of `exact_average`.
- **The transpilation.** `my_level` and `my_seed`. Fewer two-qubit gates mean fewer errors.
- **When to spend real quantum time.** Quantum time is the time a real processor spends on your jobs. The Open Plan of IBM Quantum, the plan you do not pay for, gives up to 10 minutes of it in every 28-day window (IBM Quantum documentation, "Plans overview", read on Nov 13, 2026). Tuesday's lab circuit, with the same 4,096 shots, used about 3 seconds on the IBM processor ibm_fez on Nov 13, 2026, and on Nov 14 one job with four circuits of this challenge used 7 seconds.

In [ ]:
layers = 1                     # 1, 2 or 3
my_angles = [1.20, 0.32]       # every beta first, then every gamma
my_level = 1                   # optimization level, 0 to 3
my_seed = 7                    # fixes the transpiler's random choices

assert layers in (1, 2, 3), "layers must be 1, 2 or 3."
assert len(my_angles) == 2 * layers, "my_angles needs two angles for each layer: " + str(2 * layers) + " numbers for layers = " + str(layers) + "."
worth = exact_average(my_angles, layers)
print("without noise these angles give an average cut of", round(worth, 3))
if worth < exact_average(tuesday, 1):
    print("That is less than one layer with Tuesday's angles gives (" + str(round(exact_average(tuesday, 1), 3)) + "). Look for better angles first.")

The next cell transpiles your circuit for the backend, checks it with `agreement`, and runs it. With `USE_HARDWARE = False` it is a practice run on the simulated processor, on the ring of 12 qubits. With `USE_HARDWARE = True` and `HARDWARE = "ibm"` it sends one job to the least busy IBM processor, and that run can count.

If the self-check fails, the cell stops before anything is sent.

**If your job is stuck in the queue.** Interrupt the kernel (Jupyter and qBraid Lab: Kernel menu, Interrupt Kernel. Google Colab: Runtime menu, Interrupt execution). Interrupting stops the waiting, not the job: the job stays in IBM's queue. When its turn comes it still runs and uses quantum time, and a challenge notebook has no stored results to fall back on. So cancel it before you run the cell again: run `job.cancel()` in a new cell. Then run the hardware cell again later.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
service = connect("ibm")
backend = service.least_busy(operational=True, simulator=False)
on_simulator = type(service).__name__ == "SimulatedService"      # True: the simulated processor, and nothing is sent
assert layers in (1, 2, 3) and len(my_angles) == 2 * layers, "The circuit needs 1, 2 or 3 layers and two angles for each layer."
pm = generate_preset_pass_manager(optimization_level=my_level, backend=backend, seed_transpiler=my_seed)
if on_simulator:                                # simulated processor: stay on the ring of 12 qubits
    pm = generate_preset_pass_manager(optimization_level=my_level, target=patch_target(backend, CELL),
                                      seed_transpiler=my_seed)
isa = pm.run(build_circuit(layers)).assign_parameters(my_angles)
two_qubit_gates = sum(1 for inst in isa.data if inst.operation.num_qubits == 2 and inst.operation.name != "barrier")
print("backend:", backend.name, "| two-qubit gates:", two_qubit_gates)

same = agreement(isa, my_angles, layers)
assert same > 0.999, "The transpiled circuit does not do what the QAOA circuit does (agreement " + str(round(same, 4)) + ")."

job = SamplerV2(mode=backend).run([isa], shots=SHOTS)
counts = job.result()[0].data.meas.get_counts()
run_angles, run_layers, run_device, run_practice, run_job, run_counts = list(my_angles), layers, backend.name, on_simulator, job, counts      # what this job ran
print("the circuit ran:", sum(counts.values()), "shots,", len(counts), "different bitstrings")

The next cell works out your numbers and prints the line to turn in. It prints the line only for a valid run on a real processor. It takes the layers, the angles, the device and the counts from the run itself, so changing your choices afterwards, or a run of the second option, does not change the line.

In [ ]:
from datetime import timezone

best = max(run_counts, key=lambda b: cut_value(b, edges))
print("best sample:", best, "with a cut of", cut_value(best, edges), "| average cut:", round(average_cut(run_counts, edges), 3))
trouble = problems(run_counts, run_angles, run_layers)
if run_practice:
    print("This was a practice run on the simulated processor. It is not a submission.")
    print("These samples would" + (" not be valid: " + "; ".join(trouble) if trouble else " be valid."))
elif trouble:
    print("This run is not valid: " + "; ".join(trouble))
else:
    created = run_job.creation_date.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("Valid run. Score:", score(run_counts), "points. Turn in this line:")
    print(submission_line(run_device, run_job.job_id(), run_counts, run_angles, run_layers, created))

### The second option

The cell below sends the same circuit to another quantum computer through qBraid. A circuit transpiled for an IBM processor does not fit a different machine, so the cell sends your circuit before transpiling, with your angles filled in, and the device places it on its own qubits.

How to use the cell:

1. In the Setup section set `USE_HARDWARE = True` and `HARDWARE = "qbraid"`, then run the first two setup cells again. The first time, the setup cell installs the `qbraid` package and asks for a restart.
2. Run the cell below. It prints one line for each device, with its id, its status, its number of qubits and whether it is a simulator. Then it stops with an error saying that the device `PASTE_DEVICE_ID` was not found. That stop is expected on the first run, because no device is chosen yet. Copy the id of a device that is online, is a real quantum computer and has at least 12 qubits into `QBRAID_DEVICE` in the Setup section, run that cell, then run the cell below again.
3. The cell waits for the job and prints your numbers and, for a valid run, the line to turn in. The time in that line is the time at which the job was sent.

Four lines in the cell, from `formats = ...` to `set_target_program_type`, ask the device for OpenQASM 3, the newer of two text formats in which a circuit can travel. On the tested versions the `qbraid` package can only load that one.

`wait_for_final_state()` also returns when a job has failed, and reading the counts then stops with an error. A job that is put on hold never ends the wait. In that case interrupt the kernel and ask an organizer.

The cell assumes that the device, like Qiskit, writes qubit 0 as the last character of a result. If the average it reports is close to 8.5, the value of random guessing, that assumption is the first thing to check.

The target of this challenge was set on an IBM processor. This cell was checked against the `qbraid` 0.13.0 package with the network replaced by a stand-in. It has not been run with a real qBraid account. Ask an organizer whether the second option has been confirmed for this challenge before you rely on it.

**Hardware cell, second option (qBraid).** With `USE_HARDWARE = True` and `HARDWARE = "qbraid"` this cell uses your qBraid account. With any other setting it is skipped and uses no account.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version

provider = connect("qbraid")
if provider:
    for dev in provider.get_devices():      # one line for each device
        print(dev.id, dev.status(), "| qubits:", dev.num_qubits, "| simulator:", bool(dev.profile.get("simulator")))

    device = provider.get_device(QBRAID_DEVICE)
    formats = device.profile.program_spec   # the text formats this device accepts
    formats = formats if isinstance(formats, list) else [formats]
    if "qasm3" in [item.alias for item in formats]:
        device.set_target_program_type("qasm3")

    assert layers in (1, 2, 3) and len(my_angles) == 2 * layers, "The circuit needs 1, 2 or 3 layers and two angles for each layer."
    sent_angles, sent_layers = list(my_angles), layers
    sent_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")     # the time the job is created
    job = device.run(build_circuit(sent_layers).assign_parameters(sent_angles), shots=SHOTS)   # the circuit before transpiling
    job.wait_for_final_state()
    counts = job.result().data.get_counts()

    trouble = problems(counts, sent_angles, sent_layers)
    if all(len(bits) == n for bits in counts):
        best = max(counts, key=lambda b: cut_value(b, edges))
        print("best sample:", best, "with a cut of", cut_value(best, edges), "| average cut:", round(average_cut(counts, edges), 3))
    if device.profile.get("simulator"):
        print("This device is a simulator. A run on it is not a submission.")
    elif trouble:
        print("This run is not valid: " + "; ".join(trouble))
    else:
        print("Valid run. Score:", score(counts), "points. Turn in this line:")
        print(submission_line(QBRAID_DEVICE, job.id, counts, sent_angles, sent_layers, sent_at, second_option=version("qbraid")))

## Turn it in

1. Copy the whole line that starts with `QFF26-C2`. Do not change anything in it.
2. Open the submission form: [https://forms.gle/K8Yxxn3jWMd4thas7](https://forms.gle/K8Yxxn3jWMd4thas7)
3. Enter the email you registered with and a leaderboard name, paste the line, and send. The name is shown on the leaderboards of the event page. Type the same name every time: the boards keep the name you typed first in any of the five forms, and only an organizer can change it.

You can turn in a new line whenever you have a better one. Your best valid line counts.

## If an organizer asks you to show your job

An organizer may ask you to show that a line you turned in comes from a real job in your own account. Paste the whole line into `MY_LINE` below and run the cell while they watch, in person or on a shared screen. It needs `USE_HARDWARE = True` and `HARDWARE = "ibm"` in the Setup section, and it uses the functions of "The tools". Open a fresh copy of the notebook as it was posted, restart the kernel, and run only the Setup section, the cells of "The tools" and this cell while they watch. Do not use Run All: it would also run the hardware cell and send a new job. The cell trusts what earlier cells defined in the kernel, and a fresh start shows that those are the posted ones.

The cell reads the job back from your IBM Quantum account. It prints the device, the time the job was created, the number of circuits and the shots. Then it checks the rule that the options of the Sampler stay as they are. The Sampler keeps a copy of its options with every job, and the cell compares four groups of them with the options of a new Sampler: `dynamical_decoupling` (extra pulses on qubits that wait) and `twirling` (random gates added around the gates of the circuit), which both change the circuit before it is sent, `execution`, which sets how the processor runs it, and `experimental`. A job with any of these changed fails, and the message names the option. If the job does not hold that copy, the cell prints a note that the options could not be read back, does not compare them, and goes on. Delay instructions in a circuit are not taken as a sign of changed options, because they may be part of your own circuit. It compares the circuit that was sent with the QAOA circuit for the layers and angles in your line. Then it rebuilds the submission line from the job's own counts and compares it, up to the package versions at its end, with the line you pasted. The last thing it prints is PASS or FAIL.

This check works for jobs on IBM processors. For a line from the second option, show the job in your qBraid account.

In [ ]:
MY_LINE = "PASTE_YOUR_LINE"      # the whole line you turned in

if MY_LINE == "PASTE_YOUR_LINE":
    print("Paste the line you sent into MY_LINE, then run this cell again.")
elif not (USE_HARDWARE and str(HARDWARE).strip().lower() == "ibm"):
    print('This check reads a job from your IBM Quantum account.')
    print('Set USE_HARDWARE = True and HARDWARE = "ibm" in the Setup section, run the setup cells and the cells of "The tools", then run this cell again.')
else:
    from datetime import timezone
    from qiskit_ibm_runtime import SamplerV2

    fields = dict(part.split("=", 1) for part in MY_LINE.split()[1:])
    line_layers = int(fields["layers"])
    line_angles = [float(a) for a in fields["angles"].split(",")]
    checked = connect("ibm").job(fields["job"])
    try:
        program = checked.inputs["quantum_program"]      # what was sent: the circuits and the number of shots
        sent = [item.circuit for item in program.items]
    except Exception:
        raise StopHere("FAIL: the circuits of this job could not be read back. A private job keeps none, and private jobs are not accepted.") from None
    made = checked.creation_date.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("device:", checked.backend().name, "| created (UTC):", made, "| circuits:", len(sent), "| shots:", program.shots)
    faults = []
    usual = SamplerV2(mode=SIMULATED_BACKEND).finalize_options().model_dump()     # the options of a new Sampler
    try:                                 # the options the job ran with, as the Sampler keeps them with the job
        ran_with = program.passthrough_data["post_processor"]["options"]
        changed = [group + "." + key + " = " + repr(ran_with[group].get(key))
                   for group in ("dynamical_decoupling", "twirling", "execution", "experimental")
                   for key in sorted(set(usual[group]) | set(ran_with[group]))
                   if ran_with[group].get(key) != usual[group].get(key)]
    except Exception:
        changed = []
        print("note: the options of the Sampler could not be read back, so they were not compared")
    if changed:
        faults.append("the job ran with Sampler options that are not at their defaults: " + ", ".join(changed))
    if len(sent) != 1:
        faults.append("the job must hold exactly one circuit")
    if program.shots != SHOTS:
        faults.append("the circuit needs " + str(SHOTS) + " shots")
    if not faults:
        same = agreement(sent[0], line_angles, line_layers)
        if same <= 0.999:
            faults.append("the circuit in the job is not the QAOA circuit for these layers and angles (agreement " + str(round(same, 4)) + ")")
        job_counts = checked.result()[0].data.meas.get_counts()
        faults += problems(job_counts, line_angles, line_layers)
        rebuilt = submission_line(checked.backend().name, checked.job_id(), job_counts, line_angles, line_layers, made)
        print("line rebuilt from the job:")
        print(rebuilt)
        if rebuilt.split(" qiskit=")[0] != MY_LINE.strip().split(" qiskit=")[0]:
            faults.append("the line rebuilt from the job is not the line in MY_LINE")
    print("PASS" if not faults else "FAIL: " + "; ".join(faults))

## If something breaks

| What you see | What to do |
|---|---|
| The setup cell stops and asks for a restart | Restart the kernel, then run the notebook again from the top. It installed the tested packages |
| `ValueError: Mismatching number of values and parameters` | `my_angles` needs two angles for each layer: 2, 4 or 6 numbers |
| The average is below 8.5 and no error appears | The angles are in the wrong order. Every beta comes first, then every gamma |
| "no sample is a best cut" | The samples are too noisy, or the angles are far off. Check the average without noise first. On the second option, an average close to 8.5 can mean that the device writes qubit 0 as the first character, not the last |
| `TypeError: Cannot assign object` | `my_angles` holds text. Write the angles as numbers, without quotes |
| `ValueError: Invalid optimization level` | `my_level` is 0, 1, 2 or 3 |
| `NameError: name '...' is not defined` | Run the cells in order, from the top. The cells of "Your attempt" need the cells of "The tools" |
| A practice run and a real run give different averages | Expected. The simulated processor is a snapshot of one day, and only a real run counts |
| The job stays queued | Interrupt the kernel and cancel the job: run `job.cancel()` in a new cell. Then run the hardware cell again later. A job that is not cancelled still runs and uses quantum time |
| The notebook asks for an API key | No IBM Quantum account is saved on this computer. Monday's notebook, section "Accounts", shows how to get the key |

## Where to go from here

- Every challenge stays open until Sun, Nov 22 at 11:59 PM, so you can come back to this one after the later sessions. Friday's session is about the errors that cost you points here.
- Stuck? Ask in Discord between sessions.